In [ ]:
pip install shap

In [ ]:
# ================================
# 1. Import Libraries
# ================================

import torch
import torch.nn as nn
import shap
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
from torchvision import models

# ================================
# 2. Device Configuration
# ================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================
# 3. Load ResNet50 Model
# ================================

model = models.resnet50(weights=None)

num_classes = 2
model.fc = nn.Linear(model.fc.in_features, num_classes)

model_path = "/kaggle/input/datasets/munigetiganesh/resnet/resnet50_xray_best (1).pth"

model.load_state_dict(torch.load(model_path, map_location=device))

model = model.to(device)
model.eval()

print("Model Loaded Successfully")

# ================================
# 4. Image Transform
# ================================

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# ================================
# 5. Load Chest X-ray Image
# ================================

image_path = "/kaggle/input/datasets/munigetiganesh/imagexray/person100_bacteria_475.jpeg"

image = Image.open(image_path).convert("RGB")

img_tensor = transform(image).unsqueeze(0).to(device)

# ================================
# 6. Make Prediction
# ================================

with torch.no_grad():
    output = model(img_tensor)
    probs = torch.nn.functional.softmax(output, dim=1)

prediction = torch.argmax(probs).item()

print("Prediction Class:", prediction)
print("Confidence:", probs[0][prediction].item())

# ================================
# 7. Convert Tensor → Image
# ================================

img_np = img_tensor.squeeze().cpu().numpy()
img_np = np.transpose(img_np, (1,2,0))

# Denormalize
mean = np.array([0.485,0.456,0.406])
std = np.array([0.229,0.224,0.225])

img_np = std * img_np + mean
img_np = np.clip(img_np,0,1)

# ================================
# 8. Prediction Function for SHAP
# ================================

def predict(images):

    images = torch.tensor(images).permute(0,3,1,2).float().to(device)

    with torch.no_grad():
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)

    return probs.cpu().numpy()

# ================================
# 9. SHAP Masker
# ================================

masker = shap.maskers.Image("inpaint_telea", img_np.shape)

explainer = shap.Explainer(predict, masker)

# ================================
# 10. Generate SHAP Values
# ================================

shap_values = explainer(np.expand_dims(img_np, axis=0), max_evals=500)

# ================================
# 11. Show Original Image
# ================================

plt.figure(figsize=(6,6))
plt.imshow(img_np)
plt.title("Original Chest X-ray")
plt.axis("off")
plt.show()

# ================================
# 12. SHAP Visualization
# ================================

shap.image_plot(shap_values)